# Week 3, Day 1: Matplotlib I: the anatomy of a figure

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

oris = np.arange(0, 180, 22.5)
rate = 4 + 26 * np.exp(2.2 * (np.cos(2 * np.radians(oris - 90)) - 1))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(oris, rate, marker="o")
ax.set_xlabel("orientation (deg)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title("A V1 neuron's tuning curve")
plt.show()

In [ ]:
rng = np.random.default_rng(3)
true_rate = 4 + 26 * np.exp(2.2 * (np.cos(2 * np.radians(oris - 90)) - 1))
counts = rng.poisson(true_rate[:, None] * 0.5, size=(8, 20))   # (orientations, trials)
print(counts.shape, counts[:, :5])

### Exercise: A tuning curve with error bars

Write `plot_tuning(ax, oris, counts, window)` that draws the mean firing rate (Hz) at each orientation with error bars equal to the standard error of the mean (also in Hz), labels both axes with units, and returns `ax`. Use `ax.errorbar(x, y, yerr=..., marker='o', capsize=3)`.

In [ ]:
def plot_tuning(ax, oris, counts, window):
    rate = ...
    sem = ...
    # draw with ax.errorbar, then label the axes
    return ax

fig, ax = plt.subplots(figsize=(5, 3))
plot_tuning(ax, oris, counts, window=0.5)
plt.show()

In [ ]:
import numpy as np
def test_returns_axes():
    "plot_tuning returns the Axes it drew on"
    fig, ax = plt.subplots()
    out = plot_tuning(ax, oris, counts, window=0.5)
    assert out is ax, "Return the ax you were given."
    plt.close(fig)

def test_labels():
    "both axes are labelled with units"
    fig, ax = plt.subplots()
    plot_tuning(ax, oris, counts, window=0.5)
    assert "(deg)" in ax.get_xlabel(), "The x label needs its unit, (deg)."
    assert "(Hz)" in ax.get_ylabel(), "The y label needs its unit, (Hz)."
    plt.close(fig)

def test_errorbars():
    "the curve is drawn with error bars showing the SEM in Hz"
    fig, ax = plt.subplots()
    plot_tuning(ax, oris, counts, window=0.5)
    assert len(ax.containers) >= 1, "Use ax.errorbar so the error bars are part of the plot."
    line = ax.containers[0].lines[0]
    assert np.allclose(line.get_ydata(), counts.mean(axis=1) / 0.5), "The y values should be mean counts divided by the window, in Hz."
    plt.close(fig)

test_returns_axes()
test_labels()
test_errorbars()
print("All checks passed")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
axes[0].imshow(counts, aspect="auto", cmap="viridis")
axes[0].set_xlabel("trial"); axes[0].set_ylabel("orientation index"); axes[0].set_title("counts")
plot_tuning(axes[1], oris, counts, 0.5); axes[1].set_title("tuning")
axes[2].hist(counts.ravel(), bins=range(counts.max() + 2))
axes[2].set_xlabel("spike count"); axes[2].set_ylabel("number of trials"); axes[2].set_title("all counts")
for ax, letter in zip(axes, "ABC"):
    ax.text(-0.15, 1.08, letter, transform=ax.transAxes, fontsize=13, fontweight="bold")
plt.show()

### Exercise: A two-panel figure

Make `fig` with two panels side by side that share the y axis. In the left panel plot the tuning curve for the first 10 trials only (`counts[:, :10]`); in the right, all 20 trials. Give each panel a title such as `10 trials` and `20 trials`. What changes between them?

In [ ]:
fig, axes = plt.subplots(...)
# left: first 10 trials, right: all 20
plt.show()

In [ ]:
def test_two_axes():
    "the figure has two panels side by side"
    assert len(fig.axes) == 2, f"Expected 2 axes, found {len(fig.axes)}."

def test_shared_y():
    "the panels share their y axis"
    a, b = fig.axes
    assert a.get_shared_y_axes().joined(a, b), "Use sharey=True in plt.subplots."

def test_titles():
    "each panel has a title"
    assert all(ax.get_title() for ax in fig.axes), "Give both panels a title."

test_two_axes()
test_shared_y()
test_titles()
print("All checks passed")

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(3.3, 2.4), layout="constrained")
plot_tuning(ax, oris, counts, 0.5)
fig.savefig("tuning.pdf")              # vector: for papers
fig.savefig("tuning.png", dpi=300)     # bitmap: for slides and the web
import os; print([f for f in os.listdir(".") if f.startswith("tuning")])
plt.show()
plt.rcdefaults()